In [90]:
pip install faster-whisper

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [91]:
recording_path = "c:\\temp\\Recording.m4a"
sqlite_db_path = '../db/application.db'
test_transcript = ""

In [92]:
def transcribe_audio(file_path) -> str:
    from faster_whisper import WhisperModel
    transscription = ""

    model = WhisperModel("small", device="cpu", compute_type="int8")
    segments, info = model.transcribe(file_path)

    for segment in segments:
        transscription += segment.text + " "

    return transscription.strip()

In [93]:
import sqlite3

def reset_db():
    conn = sqlite3.connect(sqlite_db_path)
    cursor = conn.cursor()

    delete_log_header_query = "DELETE FROM log_entry;"
    delete_log_segments_query = "DELETE FROM log_segment;"
    delete_log_enrichment_query = "DELETE FROM log_enrichment;"

    try:
        cursor.execute(delete_log_header_query)
        cursor.execute(delete_log_segments_query)
        cursor.execute(delete_log_enrichment_query)
        conn.commit()
        print("✅ Database reset successfully.")
    except sqlite3.Error as e:
        print(f"❌ An error occurred: {e}")
    finally:
        conn.close()

def create_log_header():
    conn = sqlite3.connect(sqlite_db_path)
    cursor = conn.cursor()

    Insert_log_entry_query = """
    INSERT INTO log_entry (entry_date, created_at, updated_at)
    VALUES ('2024-06-02', datetime('now'), datetime('now'));
    """
    try:
        cursor.execute(Insert_log_entry_query)
        new_record_id = cursor.lastrowid
        conn.commit()
    except sqlite3.Error as e:
        print(f"❌ An error occurred: {e}")
        new_record_id = None
    finally:
        conn.close()
    return new_record_id

def create_log_segment(log_entry_id, audio_filename, duration_secs, raw_transcript):
    conn = sqlite3.connect(sqlite_db_path)
    cursor = conn.cursor()

    Insert_log_entry_query = """
    INSERT INTO log_segment (log_entry_id, audio_filename, duration_secs, raw_transcript, created_at)
    VALUES (?, ?, ?, ?, datetime('now'));
    """
    try:
        cursor.execute(Insert_log_entry_query, (log_entry_id, audio_filename, duration_secs, raw_transcript))
        new_record_id = cursor.lastrowid
        conn.commit()
    except sqlite3.Error as e:
        print(f"❌ An error occurred: {e}")
        new_record_id = None
    finally:
        conn.close()

    return new_record_id

def create_log_enrichment(log_entry_id, formatted_md, followup_qs):
    conn = sqlite3.connect(sqlite_db_path)
    cursor = conn.cursor()

    Insert_log_entry_query = """
    INSERT INTO log_enrichment (log_entry_id, formatted_md, followup_qs, generated_at)
    VALUES (?, ?, ?, datetime('now'));
    """
    try:
        cursor.execute(Insert_log_entry_query, (log_entry_id, formatted_md, followup_qs))
        new_record_id = cursor.lastrowid
        conn.commit()
    except sqlite3.Error as e:
        print(f"❌ An error occurred: {e}")
        new_record_id = None
    finally:
        conn.close()
        
    return new_record_id

In [94]:
def call_llm_api(prompt: str, system: str) -> str:
    import requests

    # model is served on http://localhost:11434
    payload = {
        "model": "gemma4:e4b",
        "stream": False,
        "think": False, 
        "format": "json",
        "system": system,
        "prompt": prompt
            }
    api_response = requests.post('http://localhost:11434/api/generate',json=payload)

    import json
    returnMessages = api_response.text.splitlines()
    completeMessage = ''

    for idx, message in enumerate(returnMessages):
        returnMessage = json.loads(message)
        completeMessage += returnMessage['response']

    return completeMessage

In [95]:
markdown_prompt = """Below there will be a raw transcript of a diary like entry. Your job is to Convert the raw transcript into a neatly formatted markdown format. Use headings, bullet points, bold text, or any other formatting you think is appropriate to make the content easy to read and visually appealing.

Apply the following rules when processing the raw transcript:
  - Remove speech-to-text artefacts ("um", "uh", false starts)
  - Fix punctuation and capitalisation
  - Preserve the speaker's voice and meaning

Here the raw transscript:

"""

three_questions_prompt = """Based on the content of the diary entry, generate a list of 3 follow up questions that I can ask myself to reflect deeper on the content of the diary entry. The questions should be open ended and thought provoking. They should help me to gain deeper insights and understanding about myself based on the content of the diary entry.
Here is the content of the diary entry:
"""

In [ ]:
reset_db()
#transcribe audio:
transcription = transcribe_audio(recording_path)
if transcription == None:
    raise ValueError("Transcription is empty. Please check the audio file and transcription process.")
else:
    print("✅ Transcription completed successfully with length: " + str(len(transcription)) + " characters.")

#Create log entry header and segment:
log_entry_id = create_log_header()
if log_entry_id == None:
    raise ValueError("Failed to create log entry header. Aborting process.")
else:
    print(f"✅ Log entry header created with ID: {log_entry_id}")

log_segment_id = create_log_segment(log_entry_id, recording_path, 0, transcription)
if log_segment_id == None:
    raise ValueError("Failed to create log segment. Aborting process.")
else:
    print(f"✅ Log segment created with ID: {log_segment_id}")

#LLM calls for markdown formatting and question generation:
markdown_response = call_llm_api(markdown_prompt, transcription)
if markdown_response == None:
    raise ValueError("LLM API call for markdown formatting failed. Aborting process.")
else:        print("✅ Markdown formatting completed successfully with length: " + str(len(markdown_response)) + " characters.")

questions_response = call_llm_api(three_questions_prompt, markdown_response)
if questions_response == None:
    raise ValueError("LLM API call for question generation failed. Aborting process.")
else:
    print("✅ Question generation completed successfully with length: " + str(len(questions_response)) + " characters.")

create_log_enrichment(log_entry_id, markdown_response, questions_response)

print("🎉 End to end process completed successfully!")

✅ Database reset successfully.
✅ Transcription completed successfully with length: 646 characters.
✅ Log entry header created with ID: 1
✅ Log segment created with ID: 1
✅ Markdown formatting completed successfully with length: 664 characters.
✅ Question generation completed successfully with length: 703 characters.
🎉 End to end process completed successfully!


In [ ]:
import json
import re
import textwrap
from IPython.display import display, Markdown


def clean_text_output(text):
    if text is None:
        return "[no content]"

    cleaned = str(text).strip()

    # Unwrap JSON string values like "..." if present
    if (cleaned.startswith('"') and cleaned.endswith('"')) or (
        cleaned.startswith("'") and cleaned.endswith("'")
    ):
        try:
            cleaned = json.loads(cleaned)
        except Exception:
            pass

    # Replace escaped newline/tab artifacts with real whitespace
    cleaned = cleaned.replace('\\n', '\n').replace('\\t', '    ')
    cleaned = re.sub(r'\\+"', '"', cleaned)

    # Try to pretty-print valid JSON payloads
    try:
        parsed = json.loads(cleaned)
        return json.dumps(parsed, indent=2, ensure_ascii=False)
    except Exception:
        return cleaned


def print_section(title, text, width=88):
    print("=" * width)
    print(title)
    print("-" * width)

    cleaned = clean_text_output(text)
    for line in cleaned.splitlines():
        print(textwrap.fill(line, width=width))


print_section("Transcription", transcription)
print_section("Markdown formatted diary entry", markdown_response)
print_section("Follow up questions for self reflection", questions_response)
print("=" * 88)


Transscription:
We've had a productive day on the project, though it's not without its challenges.  The database schema is largely in place, and I'm fairly happy with the structure.  Main thing outstanding is the audio recording module.  I want to get that wired up to the transcription service by the end of the week.  Had a quick look at the whisper docs today, and I think the small model should  be more than sufficient for our needs.  One thing I keep coming back to is whether to use SQLite or move straight to Postgres.  I'm leaning towards SQLite for now, keep things simple until there's a reason to scale.  It's probably the right call. More to follow.
Markdown formatted diary entry:
{
  "raw_transcript": "We've had a productive day on the project, though it's not without its challenges. The database schema is largely in place, and I'm fairly happy with the structure. Main thing outstanding is the audio recording module. I want to get that wired up to the transcription service by the